# 4 · Precomputed candidates · the `aud:` STRING

<img src="https://redis.io/wp-content/uploads/2024/04/Logotype.svg?auto=webp&quality=85,75&width=120" alt="Redis"/>

<a href="https://colab.research.google.com/github/redis-field-engineering/redis-dsp-demo/blob/main/notebooks/04_precomputed_candidates.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Part of the Redis DSP candidate-generation demo. The full sequence walks the bid path from naive to fast across six notebooks; this is `04_precomputed_candidates.ipynb`.

**To run in Colab:** click the badge above, then *Runtime → Run all*. The setup cells below clone the repo, install dependencies, start a Redis Stack server, and load the synthetic dataset.

**To run locally:** make sure the docker-compose stack is up (`make up` from the repo root). The setup cells detect a local environment and skip the Colab-specific steps.

## Setup

These five cells prepare the environment. They are idempotent — safe to re-run, safe in either Colab or local. On Colab the first run takes about 60–90 seconds (pip install + apt install + dataset generation). Subsequent runs are near-instant because everything is cached.

In [1]:
# Setup 1/5 · clone the repo (Colab only).
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not os.path.exists("pyproject.toml"):
    print("Cloning https://github.com/redis-field-engineering/redis-dsp-demo ...")
    os.system("git clone -q https://github.com/redis-field-engineering/redis-dsp-demo.git _repo")
    os.system("cp -R _repo/. ./")
    os.system("rm -rf _repo")
    print("Repo cloned.")
elif not IN_COLAB:
    print("Local environment detected — skipping clone.")
else:
    print("Repo already present.")

Local environment detected — skipping clone.


In [2]:
# Setup 2/5 · install Python dependencies (Colab only).
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    os.system(
        f'{sys.executable} -m pip install -q '
        '"redis[hiredis]>=5.2.0" "pydantic>=2.9.0" "pandas>=2.2.0" "pyarrow>=18.0.0"'
    )
    print("Dependencies installed.")
else:
    print("Local environment detected — skipping pip install (assumes deps are already installed).")

Local environment detected — skipping pip install (assumes deps are already installed).


In [3]:
# Setup 3/5 · install and start Redis Stack (Colab only).
import os, sys, shutil
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if shutil.which("redis-stack-server") is None:
        print("Installing redis-stack-server ...")
        os.system(
            'curl -fsSL https://packages.redis.io/gpg | '
            'sudo gpg --dearmor -o /usr/share/keyrings/redis-archive-keyring.gpg'
        )
        os.system(
            'echo "deb [signed-by=/usr/share/keyrings/redis-archive-keyring.gpg] '
            'https://packages.redis.io/deb $(lsb_release -cs) main" '
            '| sudo tee /etc/apt/sources.list.d/redis.list > /dev/null'
        )
        os.system("sudo apt-get update -qq > /dev/null 2>&1")
        os.system("sudo apt-get install -qq -y redis-stack-server > /dev/null 2>&1")
    os.system("redis-stack-server --daemonize yes > /dev/null 2>&1")
    print("redis-stack-server started on :6379")
else:
    print("Local environment detected — skipping Redis install (expecting docker-compose Redis at localhost:6381).")

Local environment detected — skipping Redis install (expecting docker-compose Redis at localhost:6381).


In [4]:
# Setup 4/5 · choose the Redis URL.
import os, sys
IN_COLAB = "google.colab" in sys.modules
default_port = "6379" if IN_COLAB else "6381"
REDIS_HOST = os.getenv("REDIS_HOST", "localhost")
REDIS_PORT = os.getenv("REDIS_PORT", default_port)
REDIS_PASSWORD = os.getenv("REDIS_PASSWORD", "")
auth = f":{REDIS_PASSWORD}@" if REDIS_PASSWORD else ""
REDIS_URL = f"redis://{auth}{REDIS_HOST}:{REDIS_PORT}/0"
os.environ["DEMO_REDIS_URL"] = REDIS_URL
print(f"Redis URL: {REDIS_URL}")

Redis URL: redis://localhost:6381/0


In [5]:
# Setup 5/5 · generate and load the synthetic dataset (only if Redis is empty).
import sys, subprocess
from pathlib import Path

# Find the repo root so we can run `python -m data.synthetic` reliably.
_repo_root = Path.cwd().resolve()
while _repo_root != _repo_root.parent and not (_repo_root / "pyproject.toml").exists():
    _repo_root = _repo_root.parent
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from redis import Redis
client = Redis.from_url(REDIS_URL, decode_responses=True)
if not client.ping():
    raise RuntimeError(f"Redis at {REDIS_URL} did not answer PING")

if client.exists("meta:dataset_loaded"):
    print(
        f"Dataset already loaded: "
        f"{client.get('meta:user_count')} users, "
        f"{client.get('meta:campaign_count')} campaigns."
    )
else:
    print("Generating synthetic dataset (~30 seconds) ...")
    subprocess.run(
        [sys.executable, "-m", "data.synthetic",
         "--output", "data/generated/synthetic",
         "--num-users", "4000",
         "--num-campaigns", "2500",
         "--num-interactions", "120000",
         "--feature-count", "12"],
        cwd=_repo_root, check=True,
    )
    print("Loading dataset into Redis ...")
    subprocess.run(
        [sys.executable, "-m", "data.load_redis",
         "--redis-url", REDIS_URL,
         "--dataset-dir", "data/generated/synthetic"],
        cwd=_repo_root, check=True,
    )
    print(
        f"Done. {client.get('meta:user_count')} users, "
        f"{client.get('meta:campaign_count')} campaigns."
    )

Dataset already loaded: 4000 users, 2500 campaigns.


## Walkthrough

From here on the notebook is the demonstration.

In [6]:
# Locate the repo root so `notebooks._demo_setup` is importable regardless
# of where the kernel was launched (the package layout requires the repo
# root on sys.path).
import sys
from pathlib import Path
_repo_root = Path.cwd().resolve()
while _repo_root != _repo_root.parent and not (_repo_root / "pyproject.toml").exists():
    _repo_root = _repo_root.parent
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from notebooks._demo_setup import connect_redis, StepTimer
client = connect_redis()

connected to redis://localhost:6381/0
  users=4000  campaigns=2500  precompute_version=v17_2500_12


## What's in `aud:{maid_id}`

A single Redis STRING per MAID, containing a JSON list of campaign IDs
that survive the user's *static* targeting. "Static" here means everything
that doesn't depend on live state: geo, device, card_tier, segment
required/any_of/none_of.


In [7]:
import json
raw = client.get('aud:maid_00042')
candidates = json.loads(raw)
print(f'aud:maid_00042 has {len(candidates)} candidates')
print(f'first 10: {candidates[:10]}')
print(f'string size on the wire: {len(raw)} bytes')

aud:maid_00042 has 27 candidates
first 10: ['c01365', 'c01363', 'c01011', 'c00722', 'c00848', 'c01927', 'c02169', 'c00099', 'c02229', 'c00763']
string size on the wire: 270 bytes


## How the precompute is built

`data.hybrid_precompute._matches_static_targeting` is the function the
batch job runs against every (MAID, campaign) pair. The function is
deliberately a subset of the online filter — it leaves out everything
that can change between batch runs (pacing, budget, frequency cap,
taxonomy_filter against possibly-stale interest scores).


In [8]:
import inspect
from data import hybrid_precompute

print(inspect.getsource(hybrid_precompute._matches_static_targeting))

def _matches_static_targeting(user: UserProfile, campaign: Campaign) -> bool:
    """Match the static targeting fields used by the precompute.

    Note: the per-campaign `taxonomy_filter` is *not* evaluated here. Taxonomy
    scores can drift between batch precompute and bid time (an online
    feedback path may rewrite individual labels between batches), so the
    filter must be evaluated online against the current `maid:{maid_id}`
    interests (read via HMGET of the scoring fields). The per-MAID candidate
    list therefore over-approximates by the taxonomy_filter pass rate; the
    online taxonomy mode (`hybrid_bitmap_taxonomy`) closes that gap.
    """
    user_segments = set(user.segments)
    return (
        _matches_dimension(user.geo, campaign.geo)
        and _matches_optional_list(user.state, campaign.geo_states)
        and _matches_optional_list(user.postal_code, campaign.geo_postal_codes)
        and _matches_dimension(user.device, campaign.device)
        and _matches

The precompute is regenerated by `data.synthetic.generate_dataset` every
time the synthetic data is rebuilt. In production, the same idea applies
at a different cadence — DNA pipeline rebuilds the precompute periodically
and writes one `aud:{maid_id}` per MAID into Redis with a versioned
prefix swap.


## The precomputed_segment bid path

With the precompute in place, the bid path collapses to:

1. resolve identity,
2. fetch the hot scoring fields from `maid:` via HMGET (`user_id`, `interests_json`, `impression_count`),
3. **`GET aud:{maid_id}`** — one round trip, returns ~30 candidate IDs,
4. pipelined `HGETALL campaign:{id}` for each candidate,
5. apply minimal live gating (pacing, budget, frequency),
6. rerank.

Compare to the `full_realtime` path in notebook 2 — same steps, but
candidate generation is now a `GET` instead of a SCAN+HGETALL of all
2500 campaigns.


In [9]:
from app.models import ScoringProfile, Campaign
from app.candidate import filter_campaigns_for_user
from app.ranking import rerank_campaigns

IDENTITY_TOKEN = 'id_00042_01'
timer = StepTimer()

with timer.step('identity_resolution'):
    maid_id = client.get(f'identity:{IDENTITY_TOKEN}')

with timer.step('hot_profile_fetch'):
    payload = client.hmget(f'maid:{maid_id}', 'user_id', 'interests_json', 'impression_count')
    scoring = ScoringProfile.from_redis_hash({
        'user_id': payload[0],
        'interests_json': payload[1],
        'impression_count': payload[2],
    })

with timer.step('aud_get'):
    raw = client.get(f'aud:{maid_id}')
    candidate_ids = json.loads(raw)

with timer.step('campaign_fetch_pipelined'):
    pipe = client.pipeline(transaction=False)
    for cid in candidate_ids:
        pipe.hgetall(f'campaign:{cid}')
    campaigns = [Campaign.from_redis_hash(p) for p in pipe.execute() if p]

with timer.step('fcap_fetch'):
    fcap_counts_raw = client.hmget(f'fcap:{maid_id}', candidate_ids) if candidate_ids else []
    fcap_counts = {
        cid: int(v) for cid, v in zip(candidate_ids, fcap_counts_raw or [])
        if v is not None
    }

with timer.step('minimal_live_filter'):
    eligible = [
        c for c in campaigns
        if c.pacing_status == 'active'
        and c.spent_today_usd < c.daily_budget_usd
        and fcap_counts.get(c.campaign_id, 0) < c.frequency_cap
    ]

with timer.step('rerank'):
    top_5 = rerank_campaigns(scoring, eligible, top_k=5)

print(f'maid_id        = {maid_id}')
print(f'candidates     = {len(candidate_ids)}  (vs 2500 in full_realtime)')
print(f'eligible       = {len(eligible)}')
print(f'top 5: {[(r.campaign_id, round(r.score, 4)) for r in top_5]}')
print()
print(timer.summary())

maid_id        = maid_00042
candidates     = 27  (vs 2500 in full_realtime)
eligible       = 17
top 5: [('c01011', 5.8237), ('c00848', 4.8107), ('c01551', 4.4065), ('c01222', 4.2812), ('c02229', 4.1223)]

             identity_resolution    0.893 ms
               hot_profile_fetch    0.725 ms
                         aud_get    0.458 ms
        campaign_fetch_pipelined    3.069 ms
                      fcap_fetch    0.695 ms
             minimal_live_filter    0.008 ms
                          rerank    0.153 ms
--------------------------------------------
                           TOTAL    6.001 ms


## How this compares to the SINTER paths

| mode | candidate generation | round trips | candidates returned |
| --- | --- | ---: | ---: |
| `full_realtime` | SCAN + HGETALL all | 1 (pipelined) | 2500 |
| `maid_bruteforce_sinter` | 26 SINTER probes (sequential) | ~28 | ~50 |
| `maid_tightened_sinter` | 3 SINTER probes (pipelined) | ~3 | ~50 |
| `precomputed_segment` | `GET aud:{maid_id}` | ~3 (incl. `maid:` HMGET + fcap) | ~30 |

The precompute path is doing **less Redis work and less app work** than
either SINTER mode, because the offline batch did the SET algebra ahead of
time. On a tuned VM the precomputed_segment decision-path lands at
`~2.7 ms` p50 — about half the tightened SINTER path.


## One caveat about the precompute

The precompute deliberately leaves out anything that can change between
batch runs:

- **pacing / budget**: campaigns can run out mid-day; the bid path has to
  re-check `campaign_state:` (or use the bitmap gate from notebook 5).
- **frequency cap**: per-MAID-per-campaign counter, written every win;
  the bid path checks `fcap:{maid_id}` online.
- **taxonomy_filter**: per-campaign AND/OR/NOT on float interest scores;
  the bid path evaluates this online from the user's `interests` because
  the scores can drift between batch runs (notebook 6).

`precomputed_segment` does the minimal live gating (pacing + budget + freq).
`hybrid_precompute_plus_realtime` does the *full* live gating, including
exact targeting and the taxonomy filter. The next two notebooks tighten
the live-gating step further.
